# Task 9 — Chunking Evaluation and Selection

- Establish evaluation criteria for contract chunking.
- Compare at least two chunking methods.
- Include a 512-token chunk size with 128-token overlap.
- Measure span coverage, boundary splits, masking, extra chunks, and processing cost.
- Recommend the strongest method based on the results.
- Freeze the selected chunking settings for downstream modeling.

## Load Required Data

Load the normalized contract data and frozen train/validation split needed to evaluate chunking methods on the training set.

In [2]:
# Import libraries needed for Task 9
import pandas as pd
from pathlib import Path

# Import the shared GitHub helper function from the team's utils.py
from utils import load_file_from_github


# Base GitHub URL containing the normalized project data
# Change "main" if these files currently live on another branch.
BASE_URL = (
    "https://raw.githubusercontent.com/"
    "Break-Through-Tech/Accenture-1O-contract-review-challenge/"
    "main/"
)


# Load the normalized documents table.
# This contains the full contract text and context_group_id.
documents = pd.read_parquet(
    BASE_URL + "notebooks/split_data/documents.parquet"
)


# Load annotation sets.
# This connects contracts to CUAD categories.
annotation_sets = pd.read_parquet(
    BASE_URL + "notebooks/split_data/annotation_sets.parquet"
)


# Load the annotated clause spans.
# answer_start and answer_end are used to evaluate chunk coverage.
spans = pd.read_parquet(
    BASE_URL + "notebooks/split_data/spans.parquet"
)


# Load the CUAD category information.
categories = pd.read_parquet(
    BASE_URL + "notebooks/split_data/categories.parquet"
)


# Load the frozen train/validation split created in Task 2.
split_map = pd.read_parquet(
    BASE_URL + "notebooks/frozen-split_data/frozen-split.parquet"
)


# Verify that the required Task 9 files loaded correctly.
print("documents:", documents.shape)
print("annotation_sets:", annotation_sets.shape)
print("spans:", spans.shape)
print("categories:", categories.shape)
print("split_map:", split_map.shape)

documents: (408, 4)
annotation_sets: (16728, 4)
spans: (11180, 6)
categories: (41, 3)
split_map: (407, 2)


### Create the training-only data for Task 9

In [3]:
# Add the frozen split label to each document.
documents_with_split = documents.merge(
    split_map,
    on="context_group_id",
    how="left"
)

# Keep only documents assigned to the training split.
train_documents = documents_with_split[
    documents_with_split["split"] == "train"
].copy()

# Collect the contract IDs belonging to the training split.
train_contract_ids = set(
    train_documents["contract_id"]
)

# Keep only annotation sets belonging to training contracts.
train_annotation_sets = annotation_sets[
    annotation_sets["contract_id"].isin(train_contract_ids)
].copy()

# Keep only spans belonging to the training annotation sets.
train_spans = spans[
    spans["annotation_set_id"].isin(
        train_annotation_sets["annotation_set_id"]
    )
].copy()

# Display the final Task 9 training-data sizes.
print("Training documents:", len(train_documents))
print("Training annotation sets:", len(train_annotation_sets))
print("Training spans:", len(train_spans))

Training documents: 325
Training annotation sets: 13325
Training spans: 8964


### Prepare Evaluation Spans

Filter the training data to the 10 categories selected in Task 5 and keep only valid annotated spans. These spans will be used as the ground truth for measuring chunk coverage and boundary splits.

In [4]:

# Define the 10 CUAD categories selected in Task 5. These are the categories that Task 9 will evaluate.

selected_categories = [
    "governing_law",
    "renewal_term",
    "revenue_profit_sharing",
    "cap_on_liability",
    "uncapped_liability",
    "termination_for_convenience",
    "anti_assignment",
    "audit_rights",
    "license_grant",
    "exclusivity"
]


# Keep only annotation sets belonging to the selected: 10 categories and the training split.

selected_annotation_sets = train_annotation_sets[
    train_annotation_sets["category_id"].isin(selected_categories)
].copy()


# Keep only positive annotations. is_impossible == False means the category is actually present in the contract.

positive_annotation_sets = selected_annotation_sets[
    selected_annotation_sets["is_impossible"] == False
].copy()


# Connect the positive annotation sets to their actual annotated spans.

evaluation_spans = train_spans[
    train_spans["annotation_set_id"].isin(
        positive_annotation_sets["annotation_set_id"]
    )
].copy()

# Remove any rows that do not have valid span boundaries.
# Task 9 needs answer_start and answer_end to determine whether a chunk fully contains the annotated clause.

evaluation_spans = evaluation_spans.dropna(
    subset=["answer_start", "answer_end"]
).copy()


# Make sure the span boundary columns are integers.
evaluation_spans["answer_start"] = (
    evaluation_spans["answer_start"].astype(int)
)

evaluation_spans["answer_end"] = (
    evaluation_spans["answer_end"].astype(int)
)

# Add contract_id and category_id to each span. This makes later chunking evaluation easier.

evaluation_spans = evaluation_spans.merge(
    positive_annotation_sets[
        ["annotation_set_id", "contract_id", "category_id"]
    ],
    on="annotation_set_id",
    how="left"
)


# Display basic verification information.

print("Selected categories:", len(selected_categories))
print("Positive annotation sets:", len(positive_annotation_sets))
print("Valid evaluation spans:", len(evaluation_spans))
print(
    "Contracts represented:",
    evaluation_spans["contract_id"].nunique()
)

display(evaluation_spans.head())

Selected categories: 10
Positive annotation sets: 1536
Valid evaluation spans: 3078
Contracts represented: 296


,span_id,annotation_set_id,source_qa_id,answer_text,answer_start,answer_end,contract_id,category_id
0,contract_0001__renewal_term__span_000,contract_0001__renewal_term,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,If Distributor comp...,5543,5881,contract_0001,renewal_term
1,contract_0001__governing_law__span_000,contract_0001__governing_law,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,This Agreement is to be construed according to...,52061,52151,contract_0001,governing_law
2,contract_0001__exclusivity__span_000,contract_0001__exclusivity,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,The Company appoints the Distributor as an ex...,1854,2004,contract_0001,exclusivity
3,contract_0001__exclusivity__span_001,contract_0001__exclusivity,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Company hereby appoints ...,2112,2611,contract_0001,exclusivity
4,contract_0001__exclusivity__span_002,contract_0001__exclusivity,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,The Distributor shall not order or ...,12390,12503,contract_0001,exclusivity


In [5]:
# Check that all 10 selected categories are represented in the evaluation data.

category_span_counts = (
    evaluation_spans
    .groupby("category_id")
    .size()
    .reset_index(name="num_spans")
    .sort_values("num_spans", ascending=False)
)

display(category_span_counts)

print(
    "Number of selected categories represented:",
    evaluation_spans["category_id"].nunique()
)

,category_id,num_spans
5,license_grant,530
2,cap_on_liability,458
1,audit_rights,417
0,anti_assignment,408
4,governing_law,290
3,exclusivity,274
7,revenue_profit_sharing,269
8,termination_for_convenience,165
6,renewal_term,144
9,uncapped_liability,123


Number of selected categories represented: 10


### Establish Chunking Evaluation Criteria

Each chunking method will be evaluated using the same five criteria:

- **Span Coverage:** Percentage of annotated spans that are fully contained within at least one chunk.
- **Boundary Splits:** Number and percentage of annotated spans that are cut across chunk boundaries.
- **Masking/Loss:** Number of annotated spans that are not fully visible in any chunk and could therefore be lost during modeling.
- **Extra Chunks:** Additional chunks created compared with a non-overlapping baseline.
- **Processing Cost:** Relative amount of text/chunks the model must process.

Higher span coverage and lower boundary splits, masking, and processing cost are preferred.

In [6]:
# Define the two chunking methods for comparison.
# Method 1 is the required 512-token / 128-overlap setting.
# Method 2 uses the same chunk size without overlap so that
# we can measure whether overlap improves span preservation.

chunking_methods = {
    "512_tokens_128_overlap": {
        "chunk_size": 512,
        "overlap": 128
    },

    "512_tokens_64_overlap": {
        "chunk_size": 512,
        "overlap": 64
    },

    "512_tokens_no_overlap": {
        "chunk_size": 512,
        "overlap": 0
    }
}

# Display the settings that will be evaluated.
for method_name, settings in chunking_methods.items():
    print(
        method_name,
        "| chunk size:", settings["chunk_size"],
        "| overlap:", settings["overlap"]
    )

512_tokens_128_overlap | chunk size: 512 | overlap: 128
512_tokens_64_overlap | chunk size: 512 | overlap: 64
512_tokens_no_overlap | chunk size: 512 | overlap: 0


### Tokenize Contracts and Generate Chunks

Use the DistilRoBERTa tokenizer to split each training contract into token-based chunks while preserving character offsets. Two methods are generated:

- 512 tokens with 128-token overlap
- 512 tokens with no overlap

Character offsets allow each chunk to be compared directly with the annotated CUAD span locations.

#### <u>About DistilRoBERTa tokenizer:</u>

The DistilRoBERTa tokenizer is a tool that chops sentences into smaller pieces (like syllables or letters) so a computer can understand and process human language.

Original Sentence: "Learning is fun!"

How the tokenizer sees it: ["ĠLearning", "Ġis", "Ġfun", "!"]

(Note: The Ġ symbol simply tells the computer that a space belongs in front of that word [1].)

In [7]:
# Import the Hugging Face tokenizer used to create model-ready token chunks.
# A fast tokenizer is required because it can return character offsets.
from transformers import AutoTokenizer

TOKENIZER_NAME = "distilroberta-base"

# Load the tokenizer.
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_NAME,
    use_fast=True
)

print("Tokenizer:", TOKENIZER_NAME)
print("Model maximum length:", tokenizer.model_max_length)

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer: distilroberta-base
Model maximum length: 512


In [8]:
def create_document_chunks(documents_df, chunk_size, overlap, method_name):
    """
    Create token-based chunks manually while preserving character offsets.

    Parameters:
        documents_df: DataFrame containing contract text.
        chunk_size: Maximum total model tokens per chunk.
        overlap: Number of content tokens shared between consecutive chunks.
        method_name: Name of the chunking method.

    Returns:
        DataFrame containing one row per chunk.
    """

    # Store all generated chunks
    chunk_records = []

    # DistilRoBERTa adds special tokens such as <s> and </s>.
    # We subtract them so the final model input stays within 512 tokens.
    num_special_tokens = tokenizer.num_special_tokens_to_add(pair=False)

    # Number of actual contract-text tokens allowed in each chunk
    content_chunk_size = chunk_size - num_special_tokens

    # Determine how far forward the window moves each time
    step = content_chunk_size - overlap

    # Process each training contract separately
    for _, row in documents_df.iterrows():

        contract_id = row["contract_id"]
        context_group_id = row["context_group_id"]
        text = row["context"]

        # Tokenize the ENTIRE contract without truncating it.
        # Offset mapping connects each token back to its character location.
        encoded = tokenizer(
            text,
            add_special_tokens=False,
            truncation=False,
            return_offsets_mapping=True
        )

        token_ids = encoded["input_ids"]
        offsets = encoded["offset_mapping"]

        # Create sliding windows over the full token sequence
        chunk_number = 0

        for start_token in range(0, len(token_ids), step):

            # Find where this chunk ends
            end_token = min(
                start_token + content_chunk_size,
                len(token_ids)
            )

            # Get the character offsets for this chunk
            chunk_offsets = offsets[start_token:end_token]

            # Skip empty chunks
            if not chunk_offsets:
                continue

            # Character boundaries of the chunk
            char_start = chunk_offsets[0][0]
            char_end = chunk_offsets[-1][1]

            # Number of content tokens in this chunk
            content_tokens = end_token - start_token

            # Include the special tokens in the model-token count
            total_tokens = content_tokens + num_special_tokens

            # Save information about this chunk
            chunk_records.append({
                "method": method_name,
                "contract_id": contract_id,
                "context_group_id": context_group_id,
                "chunk_id": f"{contract_id}__chunk_{chunk_number:04d}",
                "chunk_number": chunk_number,
                "char_start": char_start,
                "char_end": char_end,
                "num_tokens": total_tokens
            })

            chunk_number += 1

            # Stop once the end of the document has been reached
            if end_token == len(token_ids):
                break

    # Convert the chunk records into a DataFrame
    return pd.DataFrame(chunk_records)

### Paragraph Chunking

In [9]:
import re
import pandas as pd


def create_paragraph_chunks(
    documents_df,
    tokenizer,
    chunk_size=512,
    long_paragraph_overlap=64,
    method_name="paragraph_based"
):
    """
    Create paragraph-aware chunks.

    Whole paragraphs are grouped together whenever possible without
    exceeding the model's token limit.

    If a single paragraph is longer than the allowed chunk size,
    that paragraph is split into token windows using a small overlap.

    Parameters:
        documents_df:
            Training documents containing contract_id,
            context_group_id, and context.

        tokenizer:
            Tokenizer used for the downstream model.

        chunk_size:
            Maximum model input length, including special tokens.

        long_paragraph_overlap:
            Overlap used only when an individual paragraph is too
            long to fit into one chunk.

        method_name:
            Name used to identify this chunking method.

    Returns:
        DataFrame containing the generated chunks and their
        character boundaries.
    """

    chunk_records = []

    # DistilRoBERTa uses special tokens around each sequence.
    num_special_tokens = tokenizer.num_special_tokens_to_add(
        pair=False
    )

    max_content_tokens = (
        chunk_size - num_special_tokens
    )

    # Process each contract separately.
    for _, row in documents_df.iterrows():

        contract_id = row["contract_id"]
        context_group_id = row["context_group_id"]
        text = str(row["context"])

        # ---------------------------------------------------------
        # Find paragraph-like sections.
        # Paragraphs are treated as blocks separated by one or more
        # blank lines.
        # Using regex the original character positions are preserved in the full contract.
        # ---------------------------------------------------------
        paragraph_matches = list(
            re.finditer(
                r"\S(?:.*?)(?=\n\s*\n|\Z)",
                text,
                flags=re.DOTALL
            )
        )

        chunk_number = 0

        # These variables keep track of paragraphs being grouped together into the current chunk.
        current_start = None
        current_end = None
        current_token_count = 0

        for match in paragraph_matches:

            paragraph_text = match.group(0)

            paragraph_start = match.start()
            paragraph_end = match.end()

            # Tokenize the paragraph without adding model special tokens.
            paragraph_encoding = tokenizer(
                paragraph_text,
                add_special_tokens=False,
                truncation=False,
                return_offsets_mapping=True
            )

            paragraph_tokens = (
                paragraph_encoding["input_ids"]
            )

            paragraph_offsets = (
                paragraph_encoding["offset_mapping"]
            )

            paragraph_token_count = len(
                paragraph_tokens
            )


            # CASE 1:
            # One paragraph is longer than the model limit.
            if paragraph_token_count > max_content_tokens:

                # Save the chunk we were already building before processing the oversized paragraph.
                if current_start is not None:

                    chunk_records.append({
                        "method": method_name,
                        "contract_id": contract_id,
                        "context_group_id": context_group_id,
                        "chunk_id":
                            f"{contract_id}__chunk_{chunk_number:04d}",
                        "chunk_number": chunk_number,
                        "char_start": current_start,
                        "char_end": current_end,
                        "num_tokens":
                            current_token_count
                            + num_special_tokens
                    })

                    chunk_number += 1

                    current_start = None
                    current_end = None
                    current_token_count = 0

                # Split the unusually long paragraph using 512-token windows with a smaller 64-token overlap.
                step = (
                    max_content_tokens
                    - long_paragraph_overlap
                )

                for start_token in range(
                    0,
                    paragraph_token_count,
                    step
                ):

                    end_token = min(
                        start_token + max_content_tokens,
                        paragraph_token_count
                    )

                    piece_offsets = paragraph_offsets[
                        start_token:end_token
                    ]

                    if not piece_offsets:
                        continue

                    # Convert paragraph-relative offsets back to character positions in the full contract.
                    char_start = (
                        paragraph_start
                        + piece_offsets[0][0]
                    )

                    char_end = (
                        paragraph_start
                        + piece_offsets[-1][1]
                    )

                    chunk_records.append({
                        "method": method_name,
                        "contract_id": contract_id,
                        "context_group_id":
                            context_group_id,
                        "chunk_id":
                            f"{contract_id}__chunk_{chunk_number:04d}",
                        "chunk_number": chunk_number,
                        "char_start": char_start,
                        "char_end": char_end,
                        "num_tokens":
                            (end_token - start_token)
                            + num_special_tokens
                    })

                    chunk_number += 1

                    # Stop once the entire paragraph has been used.
                    if end_token == paragraph_token_count:
                        break

                continue


            # CASE 2:
            # Paragraph fits inside the current chunk.
            if (
                current_token_count
                + paragraph_token_count
                <= max_content_tokens
            ):

                # If this is the first paragraph in the chunk, record its starting character.
                if current_start is None:
                    current_start = paragraph_start

                current_end = paragraph_end

                current_token_count += (
                    paragraph_token_count
                )


            # CASE 3:
            # If adding this paragraph would exceed 512 tokens, save the current chunk and begin a new chunk with this paragraph.

            else:

                chunk_records.append({
                    "method": method_name,
                    "contract_id": contract_id,
                    "context_group_id": context_group_id,
                    "chunk_id":
                        f"{contract_id}__chunk_{chunk_number:04d}",
                    "chunk_number": chunk_number,
                    "char_start": current_start,
                    "char_end": current_end,
                    "num_tokens":
                        current_token_count
                        + num_special_tokens
                })

                chunk_number += 1

                # Start a new chunk with this paragraph.
                current_start = paragraph_start
                current_end = paragraph_end
                current_token_count = (
                    paragraph_token_count
                )

        # Save the last unfinished chunk in the contract.

        if current_start is not None:

            chunk_records.append({
                "method": method_name,
                "contract_id": contract_id,
                "context_group_id": context_group_id,
                "chunk_id":
                    f"{contract_id}__chunk_{chunk_number:04d}",
                "chunk_number": chunk_number,
                "char_start": current_start,
                "char_end": current_end,
                "num_tokens":
                    current_token_count
                    + num_special_tokens
            })

    return pd.DataFrame(chunk_records)

In [10]:
# Generate 512-token chunks with 128-token overlap
chunks_overlap_128 = create_document_chunks(
    documents_df=train_documents,
    chunk_size=512,
    overlap=128,
    method_name="512_tokens_128_overlap"
)

# Generate 512-token chunks with 64-token overlap
chunks_overlap_64 = create_document_chunks(
    documents_df=train_documents,
    chunk_size=512,
    overlap=64,
    method_name="512_tokens_64_overlap"
)

# Generate 512-token chunks with no overlap
chunks_no_overlap = create_document_chunks(
    documents_df=train_documents,
    chunk_size=512,
    overlap=0,
    method_name="512_tokens_no_overlap"
)

# Generate paragraph-based chunks
chunks_paragraph = create_paragraph_chunks(
    documents_df=train_documents,
    tokenizer=tokenizer,
    chunk_size=512,
    long_paragraph_overlap=64,
    method_name="paragraph_based"
)


# Combine all four methods into one table
all_chunks = pd.concat(
    [
        chunks_overlap_128,
        chunks_overlap_64,
        chunks_no_overlap,
        chunks_paragraph
    ],
    ignore_index=True
)


# Display the total number of chunks produced
print("512 / 128 overlap chunks:", len(chunks_overlap_128))
print("512 / 64 overlap chunks:", len(chunks_overlap_64))
print("512 / no overlap chunks:", len(chunks_no_overlap))
print("Paragraph-based chunks:", len(chunks_paragraph))
print("Total chunks generated:", len(all_chunks))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (23248 > 512). Running this sequence through the model will result in indexing errors


512 / 128 overlap chunks: 11132
512 / 64 overlap chunks: 9588
512 / no overlap chunks: 8459
Paragraph-based chunks: 10841
Total chunks generated: 40020


In [11]:
# Count how many chunks each contract receives under each method
chunks_per_document = (
    all_chunks
    .groupby(
        ["method", "contract_id"]
    )
    .size()
    .reset_index(name="num_chunks")
)

# Summarize chunk counts across all training contracts
chunk_summary = (
    chunks_per_document
    .groupby("method")["num_chunks"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        max="max"
    )
    .round(2)
    .reset_index()
)

display(chunk_summary)

,method,count,mean,median,max
0,512_tokens_128_overlap,325,34.25,22.0,317
1,512_tokens_64_overlap,325,29.50,19.0,272
2,512_tokens_no_overlap,325,26.03,17.0,238
3,paragraph_based,325,33.36,21.0,260


### Measure Span Preservation

Evaluate each chunking method using the annotated clause spans.

For each method, we measure:

- **Span Coverage:** whether the full annotated span appears inside at least one chunk.
- **Boundary Splits:** spans that are cut across chunk boundaries and are not fully contained in one chunk.
- **Masking:** the percentage of a span that is missing from the best single chunk.
- **Extra Chunks:** additional chunks compared with the no-overlap baseline.
- **Processing Cost:** relative increase in the number of chunks processed.

In [12]:
# Evaluate how well one chunking method preserves the
# annotated CUAD spans.

def evaluate_chunking_method(chunks_df, spans_df, method_name):
    """
    Compare annotated spans against generated chunks.

    Returns one row per span showing:
    - whether the span is fully covered
    - whether it is split across a boundary
    - how much of the span is masked/lost from the best chunk
    """

    results = []

    # Group chunks by contract so we only compare each span with chunks from the same contract.
    chunks_by_contract = {
        contract_id: group
        for contract_id, group in chunks_df.groupby("contract_id")
    }

    # Evaluate every annotated span.
    for _, span in spans_df.iterrows():

        contract_id = span["contract_id"]
        span_start = int(span["answer_start"])
        span_end = int(span["answer_end"])

        # Calculate the annotated span length in characters.
        span_length = max(span_end - span_start, 1)

        contract_chunks = chunks_by_contract.get(contract_id)

        # Skip if no chunks were generated for this contract.
        if contract_chunks is None:
            continue

        fully_covered = False
        max_covered_chars = 0

        # Compare the span against every chunk from its contract.
        for _, chunk in contract_chunks.iterrows():

            chunk_start = int(chunk["char_start"])
            chunk_end = int(chunk["char_end"])

            # Check whether this chunk fully contains the span.
            if chunk_start <= span_start and chunk_end >= span_end:
                fully_covered = True

            # Calculate how much of the span overlaps this chunk.
            overlap_start = max(span_start, chunk_start)
            overlap_end = min(span_end, chunk_end)

            covered_chars = max(
                0,
                overlap_end - overlap_start
            )

            # Keep the largest amount of the span visible inside any single chunk.
            max_covered_chars = max(
                max_covered_chars,
                covered_chars
            )

        # Percentage of the span visible in the best single chunk.
        best_coverage_pct = (
            max_covered_chars / span_length
        ) * 100

        # Masking is the percentage of the span not visible in the best single chunk.
        masking_pct = max(
            0,
            100 - best_coverage_pct
        )

        results.append({
            "method": method_name,
            "contract_id": contract_id,
            "category_id": span["category_id"],
            "span_start": span_start,
            "span_end": span_end,
            "fully_covered": fully_covered,

            # A boundary split occurs when no single chunk contains the complete annotated span.
            "boundary_split": not fully_covered,

            "best_coverage_pct": best_coverage_pct,
            "masking_pct": masking_pct
        })

    return pd.DataFrame(results)

In [17]:
# Evaluate the 512-token / 128-overlap method.
evaluation_overlap_128 = evaluate_chunking_method(
    chunks_overlap_128,
    evaluation_spans,
    "512_tokens_128_overlap"
)

# Evaluate the 512-token / 64-overlap method.
evaluation_overlap_64 = evaluate_chunking_method(
    chunks_overlap_64,
    evaluation_spans,
    "512_tokens_64_overlap"
)

# Evaluate the 512-token / no-overlap method.
evaluation_no_overlap = evaluate_chunking_method(
    chunks_no_overlap,
    evaluation_spans,
    "512_tokens_no_overlap"
)

# Evaluate the paragraph-based chunking method.
evaluation_paragraph = evaluate_chunking_method(
    chunks_paragraph,
    evaluation_spans,
    "paragraph_based"
)

# Combine all four evaluation results.
span_evaluation = pd.concat(
    [
        evaluation_overlap_128,
        evaluation_overlap_64,
        evaluation_no_overlap,
        evaluation_paragraph
    ],
    ignore_index=True
)

# Verify the combined results.
print("Evaluation rows:", len(span_evaluation))

display(span_evaluation.head())
display(span_evaluation.tail())

Evaluation rows: 12312


,method,contract_id,category_id,span_start,span_end,fully_covered,boundary_split,best_coverage_pct,masking_pct
0,512_tokens_128_overlap,contract_0001,renewal_term,5543,5881,True,False,100.0,0.0
1,512_tokens_128_overlap,contract_0001,governing_law,52061,52151,True,False,100.0,0.0
2,512_tokens_128_overlap,contract_0001,exclusivity,1854,2004,True,False,100.0,0.0
3,512_tokens_128_overlap,contract_0001,exclusivity,2112,2611,True,False,100.0,0.0
4,512_tokens_128_overlap,contract_0001,exclusivity,12390,12503,True,False,100.0,0.0


,method,contract_id,category_id,span_start,span_end,fully_covered,boundary_split,best_coverage_pct,masking_pct
12307,paragraph_based,contract_0408,exclusivity,4335,4565,True,False,100.0,0.0
12308,paragraph_based,contract_0408,anti_assignment,22426,22555,True,False,100.0,0.0
12309,paragraph_based,contract_0408,revenue_profit_sharing,11037,11275,True,False,100.0,0.0
12310,paragraph_based,contract_0408,license_grant,3735,4334,True,False,100.0,0.0
12311,paragraph_based,contract_0408,audit_rights,12196,12388,True,False,100.0,0.0


In [18]:
# Summarize span coverage, boundary splits, and masking
# for each chunking method.

evaluation_summary = (
    span_evaluation
    .groupby("method")
    .agg(
        total_spans=("fully_covered", "size"),
        fully_covered_spans=("fully_covered", "sum"),
        boundary_splits=("boundary_split", "sum"),
        avg_masking_pct=("masking_pct", "mean")
    )
    .reset_index()
)

# Calculate coverage percentage.
evaluation_summary["span_coverage_pct"] = (
    evaluation_summary["fully_covered_spans"]
    / evaluation_summary["total_spans"]
    * 100
).round(2)

# Calculate boundary-split percentage.
evaluation_summary["boundary_split_pct"] = (
    evaluation_summary["boundary_splits"]
    / evaluation_summary["total_spans"]
    * 100
).round(2)

# Round masking percentage.
evaluation_summary["avg_masking_pct"] = (
    evaluation_summary["avg_masking_pct"]
    .round(2)
)

display(evaluation_summary)

,method,total_spans,fully_covered_spans,boundary_splits,avg_masking_pct,span_coverage_pct,boundary_split_pct
0,512_tokens_128_overlap,3078,2983,95,0.44,96.91,3.09
1,512_tokens_64_overlap,3078,2893,185,1.07,93.99,6.01
2,512_tokens_no_overlap,3078,2613,465,3.79,84.89,15.11
3,paragraph_based,3078,2990,88,0.71,97.14,2.86


### Compare Cost and Select the Final Method

Compare the additional processing cost of overlap against the improvement in span preservation. Select the method that provides the best balance between coverage and computational cost, then freeze the settings for downstream modeling.

In [19]:
# Count the total number of chunks produced by each method.
chunk_totals = pd.DataFrame({
    "method": [
        "512_tokens_128_overlap",
        "512_tokens_64_overlap",
        "512_tokens_no_overlap",
        "paragraph_based"
    ],
    "total_chunks": [
        len(chunks_overlap_128),
        len(chunks_overlap_64),
        len(chunks_no_overlap),
        len(chunks_paragraph)
    ]
})


# Use the no-overlap method as the processing-cost baseline.
baseline_chunks = len(chunks_no_overlap)


# Calculate how many additional chunks each method creates
# compared with the no-overlap baseline.
chunk_totals["extra_chunks"] = (
    chunk_totals["total_chunks"] - baseline_chunks
)


# Calculate the percentage increase in processing cost.
# More chunks means more model inputs that must be processed.
chunk_totals["processing_cost_increase_pct"] = (
    chunk_totals["extra_chunks"]
    / baseline_chunks
    * 100
).round(2)


# Combine the processing-cost results with the
# span-preservation evaluation results.
final_comparison = evaluation_summary.merge(
    chunk_totals,
    on="method",
    how="left"
)


# Arrange the columns for easier comparison.
final_comparison = final_comparison[
    [
        "method",
        "span_coverage_pct",
        "boundary_split_pct",
        "avg_masking_pct",
        "total_chunks",
        "extra_chunks",
        "processing_cost_increase_pct"
    ]
]


# Sort primarily by highest span coverage.
final_comparison = final_comparison.sort_values(
    by="span_coverage_pct",
    ascending=False
).reset_index(drop=True)


display(final_comparison)

,method,span_coverage_pct,boundary_split_pct,avg_masking_pct,total_chunks,extra_chunks,processing_cost_increase_pct
0,paragraph_based,97.14,2.86,0.71,10841,2382,28.16
1,512_tokens_128_overlap,96.91,3.09,0.44,11132,2673,31.60
2,512_tokens_64_overlap,93.99,6.01,1.07,9588,1129,13.35
3,512_tokens_no_overlap,84.89,15.11,3.79,8459,0,0.00


### Recommendation

Recommendation: I recommend paragraph-based chunking for downstream modeling. It achieved the highest span coverage at 97.14% and the lowest boundary-split rate at 2.86%, while producing 10,841 chunks. This is fewer chunks than the 512/128-overlap method and results in a lower processing-cost increase (28.16% vs. 31.60%). Although its average masking rate (0.71%) is slightly higher than 512/128 (0.44%), paragraph-based chunking provides the strongest overall balance between preserving annotated clauses, respecting natural contract structure, and controlling processing cost.